# Modern Coding Practice — Pub/Sub Broker (Amazon FAR style)

A message broker: publishers send to topics, subscribers receive what they're subscribed to. Parts
1-3 build the core (concurrency at Part 3); Parts 4-5 are stretch tasks (at-least-once delivery, then
sharding + parallel aggregation). Fill stubs, run each test cell, peek at the collapsed `ref_`
solutions only after trying.

---

## Part 1 — Topic fan-out

`Broker` with `subscribe(topic, sub)`, `unsubscribe(topic, sub)`, `publish(topic, msg)`, and
`poll(sub) -> list`. Publishing fans the message out to every current subscriber of that topic; `poll`
returns a subscriber's buffered messages **in order** and clears the buffer.

**Lock down:** delivery is per-subscriber (each gets its own copy); publishing to a topic with no
subscribers is a no-op; unsubscribing stops future (not already-buffered) messages.

In [ ]:
from collections import defaultdict


class Broker:
    def __init__(self):
        raise NotImplementedError

    def subscribe(self, topic, sub):
        raise NotImplementedError

    def unsubscribe(self, topic, sub):
        raise NotImplementedError

    def publish(self, topic, msg):
        raise NotImplementedError

    def poll(self, sub):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    b = Broker()
    b.subscribe("a", "s1"); b.subscribe("a", "s2"); b.subscribe("b", "s1")
    b.publish("a", "m1"); b.publish("b", "m2"); b.publish("c", "m3")   # c has no subscribers
    assert b.poll("s1") == ["m1", "m2"]
    assert b.poll("s2") == ["m1"]
    assert b.poll("s1") == []                                          # buffer cleared
    b.unsubscribe("a", "s1"); b.publish("a", "m4")
    assert b.poll("s1") == [] and b.poll("s2") == ["m4"]
    print("Part 1 OK")

_t1()

## Part 2 — Topic pattern matching

`topic_matches(pattern, topic) -> bool` for dotted topics (e.g. `orders.eu.new`):
- `*` matches exactly **one** segment (`orders.*` matches `orders.eu`, not `orders.eu.new` or `orders`).
- `#` matches **one or more** trailing segments and must be the last segment of the pattern
  (`orders.#` matches `orders.eu` and `orders.eu.new`, not `orders`).
- otherwise segments must match exactly.

**Lock down:** segment counts must line up for `*`; `#` is a multi-level tail wildcard. This is the
routing primitive the concurrent broker (Part 3) uses.

In [ ]:
def topic_matches(pattern, topic) -> bool:
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    m = topic_matches
    assert m("a.b", "a.b") and not m("a.b", "a.c")
    assert m("a.*", "a.b") and m("a.*", "a.x")
    assert not m("a.*", "a.b.c") and not m("a.*", "a")
    assert m("a.#", "a.b") and m("a.#", "a.b.c")
    assert not m("a.#", "a") and not m("a.#", "b.c")
    assert m("*.*", "a.b") and not m("*.*", "a")
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent broker

`ConcurrentBroker`: `subscribe(pattern, sub)`, `publish(topic, msg)` (thread-safe, routes via
`topic_matches`), and `queue(sub) -> queue.Queue` giving a subscriber's delivery queue. Many publisher
threads and subscriber threads run at once; every subscriber must receive **all** messages whose topic
matches its pattern, and a single publisher's messages must arrive **in order**.

**Discuss:** snapshot the matching subscribers under the lock, then `put` to the (already
thread-safe) `queue.Queue` outside it; per-publisher FIFO ordering vs arbitrary cross-publisher
interleaving; how a slow subscriber creates backpressure (Part 4).

In [ ]:
import queue
import threading


class ConcurrentBroker:
    def __init__(self):
        raise NotImplementedError

    def subscribe(self, pattern, sub):
        raise NotImplementedError

    def publish(self, topic, msg):
        raise NotImplementedError

    def queue(self, sub) -> "queue.Queue":
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    b = ConcurrentBroker()
    b.subscribe("orders.*", "A")        # matches orders.eu and orders.us
    b.subscribe("orders.eu", "B")       # matches only orders.eu
    P, M = 4, 50
    topics = ["orders.eu", "orders.us"]
    sentinel = object()
    received, rlock = {}, threading.Lock()

    def consumer(sub):
        q, got = b.queue(sub), []
        while True:
            item = q.get()
            if item is sentinel:
                break
            got.append(item)
        with rlock:
            received[sub] = got

    ca = threading.Thread(target=consumer, args=("A",)); ca.start()
    cb = threading.Thread(target=consumer, args=("B",)); cb.start()

    def publisher(pid):
        for i in range(M):
            b.publish(topics[i % 2], (pid, i))

    pubs = [threading.Thread(target=publisher, args=(p,)) for p in range(P)]
    for t in pubs: t.start()
    for t in pubs: t.join()
    b.queue("A").put(sentinel); b.queue("B").put(sentinel)
    ca.join(); cb.join()

    assert len(received["A"]) == P * M                 # A sees every message
    assert len(received["B"]) == P * M // 2            # B sees only orders.eu (the even i)
    assert all(t == "orders.eu" for t, _ in received["B"])
    for pid in range(P):                               # per-publisher order preserved
        seqs = [i for (t, (p, i)) in received["A"] if p == pid]
        assert seqs == sorted(seqs)
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — At-least-once delivery: ack, redeliver, dedup

Reliable delivery needs acknowledgements. Build `ReliableBroker`:
- `publish(topic, msg) -> msg_id`: assign a monotonic id and deliver `(msg_id, msg)` to matching
  subscribers' **pending** (unacked) set.
- `poll(sub) -> list[(msg_id, msg)]`: the subscriber's currently-unacked messages, in order.
- `ack(sub, msg_id)`: remove an acknowledged message.
- `redeliver(sub) -> list[(msg_id, msg)]`: messages still unacked (what a timeout would resend).

Because redelivery can produce duplicates, also implement `apply_dedup(stream) -> list`: drop repeated
`msg_id`s, keeping the first occurrence (consumer-side idempotency).

**Discuss:** at-least-once vs exactly-once, why dedup lives on the consumer, and visibility-timeout
redelivery (SQS-style).

In [ ]:
class ReliableBroker:
    def __init__(self):
        raise NotImplementedError

    def subscribe(self, pattern, sub):
        raise NotImplementedError

    def publish(self, topic, msg):
        raise NotImplementedError

    def poll(self, sub):
        raise NotImplementedError

    def ack(self, sub, msg_id):
        raise NotImplementedError

    def redeliver(self, sub):
        raise NotImplementedError


def apply_dedup(stream):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    b = ReliableBroker()
    b.subscribe("t", "s")
    id1 = b.publish("t", "m1"); id2 = b.publish("t", "m2")
    assert [m for _, m in b.poll("s")] == ["m1", "m2"]
    b.ack("s", id1)
    assert [i for i, _ in b.redeliver("s")] == [id2]          # m1 acked, m2 still pending
    b.ack("s", id2)
    assert b.redeliver("s") == []
    dupd = [(id1, "m1"), (id2, "m2"), (id2, "m2")]            # redelivery created a duplicate
    assert apply_dedup(dupd) == [(id1, "m1"), (id2, "m2")]
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Sharded broker + parallel aggregation

**(a)** `ShardedBroker(n_shards)` partitions topics across `n_shards` independent sub-brokers (each
with its own lock) to cut lock contention: `subscribe(topic, sub)`, `publish(topic, msg)`, and
`poll(sub)` (which gathers across shards). Route by `hash(topic) % n_shards`.

**(b)** `parallel_topic_counts(log, n_procs) -> dict`: count messages per topic over a large recorded
log (CPU-bound), across processes with `ProcessPoolExecutor`; the worker is
`pubsub_workers.count_topics`.

**Discuss:** sharding vs one global lock, hot-topic skew, GIL (counting is CPU-bound -> processes),
and combiner associativity for the merge.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import pubsub_workers


class ShardedBroker:
    def __init__(self, n_shards):
        raise NotImplementedError

    def subscribe(self, topic, sub):
        raise NotImplementedError

    def publish(self, topic, msg):
        raise NotImplementedError

    def poll(self, sub):
        raise NotImplementedError


def parallel_topic_counts(log, n_procs) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    b = ShardedBroker(4)
    b.subscribe("a", "s1"); b.subscribe("b", "s1")
    b.publish("a", "m1"); b.publish("b", "m2")
    assert set(b.poll("s1")) == {"m1", "m2"}
    log = ["a", "b", "a", "c", "a", "b"]
    assert parallel_topic_counts(log, 2) == {"a": 3, "b": 2, "c": 1}
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Durable subscriptions / persistent log (Kafka-style offsets) vs ephemeral queues.
- Ordering guarantees: per-partition vs global; consumer groups & rebalancing.
- Flow control: bounded queues, drop policies, slow-consumer detection.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
from collections import defaultdict
import queue
import threading
from concurrent.futures import ProcessPoolExecutor
import pubsub_workers


class RefBroker:
    def __init__(self):
        self._subs = defaultdict(set)
        self._buf = defaultdict(list)

    def subscribe(self, topic, sub):
        self._subs[topic].add(sub)

    def unsubscribe(self, topic, sub):
        self._subs[topic].discard(sub)

    def publish(self, topic, msg):
        for s in self._subs.get(topic, ()):
            self._buf[s].append(msg)

    def poll(self, sub):
        out, self._buf[sub] = self._buf[sub], []
        return out


def ref_topic_matches(pattern, topic):
    p, t = pattern.split("."), topic.split(".")
    for i, seg in enumerate(p):
        if seg == "#":
            return i == len(p) - 1 and len(t) > i      # tail wildcard, >=1 remaining
        if i >= len(t):
            return False
        if seg != "*" and seg != t[i]:
            return False
    return len(t) == len(p)


class RefConcurrentBroker:
    def __init__(self):
        self._subs = []                                # (pattern, sub)
        self._queues = defaultdict(queue.Queue)
        self._lock = threading.Lock()

    def subscribe(self, pattern, sub):
        with self._lock:
            self._subs.append((pattern, sub))
            _ = self._queues[sub]                      # ensure queue exists

    def publish(self, topic, msg):
        with self._lock:                               # snapshot targets under lock
            targets = [s for pat, s in self._subs if ref_topic_matches(pat, topic)]
        for s in targets:                              # queue.Queue.put is itself thread-safe
            self._queues[s].put((topic, msg))

    def queue(self, sub):
        return self._queues[sub]


class RefReliableBroker:
    def __init__(self):
        self._subs = []
        self._pending = defaultdict(dict)              # sub -> {msg_id: msg} (insertion-ordered)
        self._next = 0

    def subscribe(self, pattern, sub):
        self._subs.append((pattern, sub))

    def publish(self, topic, msg):
        mid = self._next
        self._next += 1
        for pat, s in self._subs:
            if ref_topic_matches(pat, topic):
                self._pending[s][mid] = msg
        return mid

    def poll(self, sub):
        return list(self._pending[sub].items())

    def ack(self, sub, msg_id):
        self._pending[sub].pop(msg_id, None)

    def redeliver(self, sub):
        return list(self._pending[sub].items())


def ref_apply_dedup(stream):
    seen, out = set(), []
    for mid, msg in stream:
        if mid in seen:
            continue
        seen.add(mid)
        out.append((mid, msg))
    return out


class RefShardedBroker:
    def __init__(self, n_shards):
        self.n = n_shards
        self.shards = [RefBroker() for _ in range(n_shards)]
        self.locks = [threading.Lock() for _ in range(n_shards)]

    def _idx(self, topic):
        return hash(topic) % self.n

    def subscribe(self, topic, sub):
        i = self._idx(topic)
        with self.locks[i]:
            self.shards[i].subscribe(topic, sub)

    def publish(self, topic, msg):
        i = self._idx(topic)
        with self.locks[i]:
            self.shards[i].publish(topic, msg)

    def poll(self, sub):
        out = []
        for i in range(self.n):
            with self.locks[i]:
                out += self.shards[i].poll(sub)
        return out


def ref_parallel_topic_counts(log, n_procs):
    chunks = [log[i::n_procs] for i in range(n_procs)]
    total = {}
    with ProcessPoolExecutor(max_workers=n_procs) as ex:
        for d in ex.map(pubsub_workers.count_topics, chunks):
            for k, v in d.items():
                total[k] = total.get(k, 0) + v
    return total


_b = RefBroker(); _b.subscribe("x", "s"); _b.publish("x", 1); assert _b.poll("s") == [1]
assert ref_topic_matches("a.#", "a.b.c") and not ref_topic_matches("a.*", "a.b.c")
_r = RefReliableBroker(); _r.subscribe("t", "s"); _i = _r.publish("t", "m")
assert _r.redeliver("s") == [(_i, "m")]; _r.ack("s", _i); assert _r.redeliver("s") == []
assert ref_apply_dedup([(0, "a"), (0, "a"), (1, "b")]) == [(0, "a"), (1, "b")]
assert ref_parallel_topic_counts(["a", "a", "b"], 2) == {"a": 2, "b": 1}
print("reference solutions OK")